# 📊 Systematic Review Analysis: ML for Childhood Nutritional Outcomes

**Objetivo**: Analisar dados extraídos de artigos sobre machine learning para predição de desfechos nutricionais em crianças (0-11 anos)

**Outcomes incluídos**: Stunting, Underweight, Overweight, Obesity

**Autora**: Isa (Doutorado em Nutrição em Saúde Pública - USP)

## 1. Setup e Configuração

In [ ]:
# Instalar dependências (se necessário)
!pip install openpyxl pandas numpy matplotlib seaborn plotly -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

# Configurações
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
plt.style.use('seaborn-v0_8-whitegrid')

print("✅ Setup completo!")

## 2. Carregar Dados

**Instruções**:
1. Faça upload do arquivo `extracao_artigo1_v2.xlsx` no Colab
2. Ou monte o Google Drive se o arquivo estiver lá

In [ ]:
# Opção 1: Upload direto
from google.colab import files
uploaded = files.upload()

# Pegar o nome do arquivo
filename = list(uploaded.keys())[0]
print(f"Arquivo carregado: {filename}")

In [ ]:
# Opção 2: Google Drive (descomente se preferir)
# from google.colab import drive
# drive.mount('/content/drive')
# filename = '/content/drive/MyDrive/SEU_CAMINHO/extracao_artigo1_v2.xlsx'

In [ ]:
# Carregar dados (header na linha 1 = index 1)
df = pd.read_excel(filename, header=1)

# Remover coluna vazia se existir
df = df.loc[:, ~df.columns.str.contains('Unnamed')]

print(f"Shape: {df.shape}")
print(f"Artigos únicos: {df['ID'].nunique()}")
df.head()

## 3. Limpeza de Dados

In [ ]:
# Verificar outcomes
print("📊 Distribuição de Outcomes (antes da limpeza):")
print(df['Outcome'].value_counts())

In [ ]:
# Outcomes válidos para a revisão
VALID_OUTCOMES = ['Stunting', 'Underweight', 'Overweight', 'Obesity']

# Identificar artigos a excluir
df_excluir = df[~df['Outcome'].isin(VALID_OUTCOMES)]
print("🚫 Artigos a EXCLUIR:")
print(df_excluir[['ID', 'Outcome', 'First Author']].to_string())

# Filtrar apenas outcomes válidos
df_clean = df[df['Outcome'].isin(VALID_OUTCOMES)].copy()
print(f"\n✅ Após limpeza: {len(df_clean)} linhas, {df_clean['ID'].nunique()} artigos")

## 4. Análise de Data Leakage 🚨

In [ ]:
def detect_data_leakage(row):
    """Detecta data leakage com base no outcome e preditores"""
    outcome = str(row['Outcome']).lower()
    predictors = str(row.get('Top 5 Predictors', '')).lower() + ' ' + str(row.get('All Variables Used', '')).lower()
    
    issues = []
    
    # Stunting + height
    if 'stunting' in outcome:
        if any(x in predictors for x in ['height', 'haz', 'hfa', 'h/a', 'height-for-age', 'tb/u']):
            issues.append('HEIGHT→STUNTING')
    
    # Underweight + weight
    if 'underweight' in outcome:
        if any(x in predictors for x in ['weight', 'waz', 'wfa', 'w/a', 'weight-for-age', 'bb/u']):
            issues.append('WEIGHT→UNDERWEIGHT')
    
    # Obesity/Overweight + BMI
    if 'obesity' in outcome or 'overweight' in outcome:
        if any(x in predictors for x in ['bmi', 'baz', 'body mass', 'imc']):
            issues.append('BMI→OBESITY/OVERWEIGHT')
    
    return '; '.join(issues) if issues else None

# Aplicar detecção
df_clean['Data_Leakage'] = df_clean.apply(detect_data_leakage, axis=1)

# Resumo
leakage_count = df_clean['Data_Leakage'].notna().sum()
print(f"🚨 DATA LEAKAGE IDENTIFICADO: {leakage_count} de {len(df_clean)} linhas ({leakage_count/len(df_clean)*100:.1f}%)")

# Mostrar afetados
df_leakage = df_clean[df_clean['Data_Leakage'].notna()]
print(f"\nArtigos afetados:")
print(df_leakage[['ID', 'Outcome', 'Data_Leakage', 'AUC']].to_string())

## 5. Estatísticas Descritivas

In [ ]:
# Função para converter AUC para numérico
def parse_auc(val):
    if pd.isna(val) or str(val).lower() in ['not reported', 'nan', 'nr']:
        return np.nan
    match = re.search(r'(\d+\.?\d*)', str(val))
    return float(match.group(1)) if match else np.nan

df_clean['AUC_num'] = df_clean['AUC'].apply(parse_auc)

print("📊 RESUMO POR OUTCOME")
print("=" * 60)

for outcome in VALID_OUTCOMES:
    subset = df_clean[df_clean['Outcome'] == outcome]
    if len(subset) == 0:
        continue
    
    auc_vals = subset['AUC_num'].dropna()
    leakage = subset['Data_Leakage'].notna().sum()
    
    print(f"\n🎯 {outcome}")
    print(f"   N estudos: {len(subset)}")
    print(f"   Data leakage: {leakage} ({leakage/len(subset)*100:.0f}%)")
    if len(auc_vals) > 0:
        print(f"   AUC: {auc_vals.min():.2f} - {auc_vals.max():.2f} (mediana: {auc_vals.median():.2f})")

In [ ]:
# Distribuição geográfica
print("🌍 DISTRIBUIÇÃO GEOGRÁFICA")
print("=" * 40)
print(df_clean['Country/Region'].value_counts())

In [ ]:
# Algoritmos
print("🤖 ALGORITMOS MAIS USADOS")
print("=" * 40)
print(df_clean['Main Algorithm'].value_counts())

## 6. Análise PROBAST

In [ ]:
# PROBAST Overall
print("⚠️ PROBAST - OVERALL RISK OF BIAS")
print("=" * 40)
probast_overall = df_clean['PROBAST: Overall Risk of Bias'].value_counts()
print(probast_overall)

# Calcular porcentagens
total = len(df_clean)
for rating, count in probast_overall.items():
    print(f"  {rating}: {count/total*100:.1f}%")

In [ ]:
# PROBAST por domínio
probast_cols = {
    'Participants': 'PROBAST-P1: Risk of Bias',
    'Predictors': 'PROBAST-P2: Risk of Bias',
    'Outcome': 'PROBAST-O3: Risk of Bias',
    'Analysis': 'PROBAST-A4: Risk of Bias'
}

print("📋 PROBAST POR DOMÍNIO")
print("=" * 60)

probast_summary = []
for domain, col in probast_cols.items():
    if col in df_clean.columns:
        counts = df_clean[col].value_counts()
        low = counts.get('LOW', 0)
        high = counts.get('HIGH', 0)
        unclear = counts.get('Unclear', 0) + counts.get('UNCLEAR', 0)
        
        probast_summary.append({
            'Domain': domain,
            'LOW (%)': f"{low} ({low/total*100:.0f}%)",
            'HIGH (%)': f"{high} ({high/total*100:.0f}%)",
            'UNCLEAR (%)': f"{unclear} ({unclear/total*100:.0f}%)"
        })

pd.DataFrame(probast_summary)

## 7. Visualizações

In [ ]:
# Gráfico 1: Distribuição de Outcomes
fig, ax = plt.subplots(figsize=(10, 6))
outcome_counts = df_clean['Outcome'].value_counts()
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
outcome_counts.plot(kind='bar', color=colors, ax=ax)
ax.set_title('Distribuição de Estudos por Outcome', fontsize=14)
ax.set_xlabel('Outcome')
ax.set_ylabel('Número de Estudos')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(outcome_counts):
    ax.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_outcomes.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Gráfico 2: AUC por Outcome (boxplot)
fig, ax = plt.subplots(figsize=(10, 6))
df_plot = df_clean[df_clean['AUC_num'].notna()]

sns.boxplot(data=df_plot, x='Outcome', y='AUC_num', palette='Set2', ax=ax)
sns.stripplot(data=df_plot, x='Outcome', y='AUC_num', color='black', alpha=0.5, ax=ax)

ax.set_title('Distribuição de AUC por Outcome', fontsize=14)
ax.set_xlabel('Outcome')
ax.set_ylabel('AUC')
ax.axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='AUC = 0.70')
ax.legend()
plt.tight_layout()
plt.savefig('fig2_auc_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Gráfico 3: PROBAST Risk of Bias
fig, ax = plt.subplots(figsize=(8, 6))
probast_counts = df_clean['PROBAST: Overall Risk of Bias'].value_counts()
colors_probast = {'HIGH': '#e74c3c', 'LOW': '#2ecc71', 'Unclear': '#f39c12'}
probast_counts.plot(kind='pie', autopct='%1.1f%%', colors=[colors_probast.get(x, '#95a5a6') for x in probast_counts.index], ax=ax)
ax.set_title('PROBAST - Overall Risk of Bias', fontsize=14)
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('fig3_probast_pie.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Gráfico 4: Algoritmos (top 10)
fig, ax = plt.subplots(figsize=(12, 6))
algo_counts = df_clean['Main Algorithm'].value_counts().head(10)
algo_counts.plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Top 10 Algoritmos de Machine Learning', fontsize=14)
ax.set_xlabel('Número de Estudos')
ax.invert_yaxis()
for i, v in enumerate(algo_counts):
    ax.text(v + 0.2, i, str(v), va='center')
plt.tight_layout()
plt.savefig('fig4_algorithms.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Análise Separada: COM vs SEM Data Leakage

In [ ]:
# Dividir datasets
df_no_leakage = df_clean[df_clean['Data_Leakage'].isna()]
df_with_leakage = df_clean[df_clean['Data_Leakage'].notna()]

print("📊 COMPARAÇÃO: COM vs SEM DATA LEAKAGE")
print("=" * 60)

print(f"\n✅ SEM Data Leakage: {len(df_no_leakage)} estudos")
if len(df_no_leakage) > 0:
    auc_no = df_no_leakage['AUC_num'].dropna()
    print(f"   AUC range: {auc_no.min():.2f} - {auc_no.max():.2f} (mediana: {auc_no.median():.2f})")

print(f"\n🚨 COM Data Leakage: {len(df_with_leakage)} estudos")
if len(df_with_leakage) > 0:
    auc_with = df_with_leakage['AUC_num'].dropna()
    print(f"   AUC range: {auc_with.min():.2f} - {auc_with.max():.2f} (mediana: {auc_with.median():.2f})")

## 9. Gerar Tabelas para Publicação

In [ ]:
# Tabela 1: Características dos Estudos
cols_table1 = ['ID', 'First Author', 'Year', 'Country/Region', 'N (sample)', 
               'Age (months)', 'Outcome', 'Main Algorithm', 'AUC', 
               'External Validation?', 'PROBAST: Overall Risk of Bias', 'Data_Leakage']

table1 = df_clean[cols_table1].copy()
table1.to_excel('Table1_Study_Characteristics.xlsx', index=False)
print("✅ Tabela 1 salva: Table1_Study_Characteristics.xlsx")
table1.head(10)

In [ ]:
# Tabela 2: Performance por Outcome
table2_data = []
for outcome in VALID_OUTCOMES:
    subset = df_clean[df_clean['Outcome'] == outcome]
    if len(subset) == 0:
        continue
    auc_vals = subset['AUC_num'].dropna()
    ext_val = (subset['External Validation?'] == 'Yes').sum()
    
    table2_data.append({
        'Outcome': outcome,
        'N Studies': len(subset),
        'N with AUC': len(auc_vals),
        'AUC Range': f"{auc_vals.min():.2f} - {auc_vals.max():.2f}" if len(auc_vals) > 0 else 'N/A',
        'AUC Median (IQR)': f"{auc_vals.median():.2f} ({auc_vals.quantile(0.25):.2f}-{auc_vals.quantile(0.75):.2f})" if len(auc_vals) > 0 else 'N/A',
        'External Validation (%)': f"{ext_val} ({ext_val/len(subset)*100:.0f}%)"
    })

table2 = pd.DataFrame(table2_data)
table2.to_excel('Table2_Performance_by_Outcome.xlsx', index=False)
print("✅ Tabela 2 salva: Table2_Performance_by_Outcome.xlsx")
table2

## 10. Exportar Resultados

In [ ]:
# Salvar dataset limpo
df_clean.to_excel('extraction_data_clean.xlsx', index=False)
print("✅ Dataset limpo salvo: extraction_data_clean.xlsx")

# Baixar arquivos (Colab)
from google.colab import files
files.download('Table1_Study_Characteristics.xlsx')
files.download('Table2_Performance_by_Outcome.xlsx')
files.download('extraction_data_clean.xlsx')
files.download('fig1_outcomes.png')
files.download('fig2_auc_boxplot.png')
files.download('fig3_probast_pie.png')
files.download('fig4_algorithms.png')

---
## 📋 Próximos Passos

1. **Revisar** artigos com data leakage e decidir como reportar
2. **Localizar** AA_31 (artigo faltando)
3. **Decidir** sobre AA_10 e AA_12 (outcome genérico)
4. **Completar** números do PRISMA flow
5. **Inserir** data da busca e número PROSPERO no manuscrito